# 03 Run New Models Complete

This single notebook does the full new-model workflow:

1. Loads the prepared panel data.
2. Runs the Naive / Random Walk baseline.
3. Runs or loads N-BEATS and PatchTST results.
4. Runs or loads Chronos zero-shot results.
5. Runs or loads Kronos zero-shot results.
6. Merges all predictions.
7. Creates summaries, cross-sectional RankIC, and plots.

Use this instead of running a separate results-loader notebook.

In [1]:
from pathlib import Path
import sys
import warnings

# Works whether the notebook is inside the project root or inside a notebooks/ folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)

warnings.filterwarnings("ignore", message="val_check_steps is greater than max_steps.*")

Project root: c:\Users\user\OneDrive\Desktop\Praktikum\Final-Project\new_equity_forecasting_project
Python: c:\Users\user\AppData\Local\Programs\Python\Python311\python.exe


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from src.config import (
    DATA_PROCESSED_DIR,
    RESULTS_DIR,
    PLOTS_DIR,
    HORIZONS,
    INPUT_SIZE,
    TEST_STEP,
    MAX_TEST_DATES,
    MAX_STEPS,
    FREQ,
    MIN_ASSETS_FOR_RANKIC,
    RANDOM_SEED,
    CHRONOS_MODEL_ID,
    KRONOS_MODEL_ID,
    KRONOS_TOKENIZER_ID,
    KRONOS_LOCAL_REPO,
)

from src.new_models import (
    BenchmarkConfig,
    run_neuralforecast_models,
    run_chronos_zero_shot,
    run_kronos_zero_shot,
    run_naive_baseline,
    run_classical_models,
)

from src.data_utils import load_prepared_panel

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print("Results dir:", RESULTS_DIR)
print("Plots dir:", PLOTS_DIR)

Results dir: C:\Users\user\OneDrive\Desktop\Praktikum\Final-Project\new_equity_forecasting_project\results\new_models
Plots dir: C:\Users\user\OneDrive\Desktop\Praktikum\Final-Project\new_equity_forecasting_project\results\new_models\plots


In [3]:
# Main run controls
# Set FORCE_RERUN_* = False to reuse existing CSV files and avoid retraining.

RUN_NEURAL_MODELS = True
RUN_CHRONOS = True
RUN_KRONOS = True

FORCE_RERUN_NEURAL = True
FORCE_RERUN_CHRONOS = True
FORCE_RERUN_KRONOS = True

# Keep this True while setting up Kronos. It attempts Kronos, but still lets the notebook continue
# if the local Kronos repo/dependencies are not ready yet.
ALLOW_CONTINUE_IF_KRONOS_FAILS = True

NEURAL_MODELS_TO_RUN = ["NBEATS", "PatchTST"]

# For faster debugging, uncomment this smaller config.
# OVERRIDE_CONFIG = dict(horizons=[1], input_size=60, test_step=63, max_test_dates=3, max_steps=10)

OVERRIDE_CONFIG = None

In [4]:
panel = load_prepared_panel(DATA_PROCESSED_DIR)
panel = panel.sort_values(["Ticker", "Date"]).reset_index(drop=True)

if OVERRIDE_CONFIG:
    config = BenchmarkConfig(
        horizons=OVERRIDE_CONFIG.get("horizons", HORIZONS),
        input_size=OVERRIDE_CONFIG.get("input_size", INPUT_SIZE),
        test_step=OVERRIDE_CONFIG.get("test_step", TEST_STEP),
        max_test_dates=OVERRIDE_CONFIG.get("max_test_dates", MAX_TEST_DATES),
        max_steps=OVERRIDE_CONFIG.get("max_steps", MAX_STEPS),
        freq=FREQ,
        random_seed=RANDOM_SEED,
    )
else:
    config = BenchmarkConfig(
        horizons=HORIZONS,
        input_size=INPUT_SIZE,
        test_step=TEST_STEP,
        max_test_dates=MAX_TEST_DATES,
        max_steps=MAX_STEPS,
        freq=FREQ,
        random_seed=RANDOM_SEED,
    )

print("Panel shape:", panel.shape)
print("Tickers:", panel["Ticker"].nunique(), sorted(panel["Ticker"].unique())[:20])
print("Date range:", panel["Date"].min(), "to", panel["Date"].max())
print("Config:", config)
display(panel.head())

Panel shape: (461532, 11)
Tickers: 243 ['A', 'AAPL', 'ABBV', 'ABNB', 'ABT', 'ACN', 'ADBE', 'ADI', 'AIG', 'ALL', 'AMAT', 'AMD', 'AMGN', 'AMT', 'AMZN', 'APD', 'APO', 'ARES', 'AUDUSD=X', 'AVB']
Date range: 2019-01-01 00:00:00 to 2026-05-01 00:00:00
Config: BenchmarkConfig(horizons=[1, 5], input_size=128, test_step=21, max_test_dates=18, max_steps=40, freq='B', random_seed=42)


,Date,Ticker,open,high,low,close,adj_close,volume,return_1d,log_return_1d,amount
0,2019-01-02,A,66.500000,66.570000,65.300003,65.690002,62.300140,2113300.0,NaN,NaN,1.388227e+08
1,2019-01-03,A,65.529999,65.779999,62.000000,63.270000,60.005016,5383900.0,-0.036840,-0.037535,3.406394e+08
2,2019-01-04,A,64.089996,65.949997,64.089996,65.459999,62.082012,3123700.0,0.034614,0.034028,2.044774e+08
3,2019-01-07,A,65.639999,67.430000,65.610001,66.849998,63.400280,3235100.0,0.021234,0.021012,2.162664e+08
4,2019-01-08,A,67.589996,68.209999,66.699997,67.830002,64.329704,1578100.0,0.014660,0.014553,1.070425e+08


## Helper functions for summaries, RankIC, and plots

These are included directly in the notebook so the file can still summarize results even if `src.evaluation` has version issues.

In [5]:
def summarize_predictions_local(predictions: pd.DataFrame) -> pd.DataFrame:
    rows = []
    required = ["Model", "Horizon", "y_true", "y_pred"]
    missing = [c for c in required if c not in predictions.columns]
    if missing:
        raise ValueError(f"Missing columns for summary: {missing}")

    for (model, horizon), g in predictions.groupby(["Model", "Horizon"]):
        g = g.dropna(subset=["y_true", "y_pred"])
        if g.empty:
            continue
        err = g["y_true"].to_numpy() - g["y_pred"].to_numpy()
        rows.append({
            "Model": model,
            "Horizon": int(horizon),
            "N": int(len(g)),
            "MAE": float(np.mean(np.abs(err))),
            "RMSE": float(np.sqrt(np.mean(err ** 2))),
            "DirectionalAccuracy": float(np.mean(np.sign(g["y_true"]) == np.sign(g["y_pred"]))),
            "MeanActualReturn": float(g["y_true"].mean()),
            "MeanPredReturn": float(g["y_pred"].mean()),
        })
    if not rows:
        return pd.DataFrame(columns=["Model", "Horizon", "N", "MAE", "RMSE", "DirectionalAccuracy", "MeanActualReturn", "MeanPredReturn"])
    return pd.DataFrame(rows).sort_values(["Horizon", "MAE", "Model"]).reset_index(drop=True)


def cross_sectional_rank_ic_local(predictions: pd.DataFrame, min_assets: int = 5) -> pd.DataFrame:
    rows = []
    required = ["Date", "Ticker", "Model", "Horizon", "y_true", "y_pred"]
    missing = [c for c in required if c not in predictions.columns]
    if missing:
        raise ValueError(f"Missing columns for RankIC: {missing}")

    for (model, horizon, date), g in predictions.groupby(["Model", "Horizon", "Date"]):
        g = g.dropna(subset=["y_true", "y_pred"])
        if g["Ticker"].nunique() < min_assets:
            continue
        if g["y_true"].nunique() < 2 or g["y_pred"].nunique() < 2:
            continue
        ic, _ = spearmanr(g["y_pred"], g["y_true"])
        if np.isfinite(ic):
            rows.append({
                "Model": model,
                "Horizon": int(horizon),
                "Date": pd.Timestamp(date),
                "RankIC": float(ic),
                "N_Assets": int(g["Ticker"].nunique()),
            })
    return pd.DataFrame(rows)


def summarize_rank_ic_local(rank_ic_df: pd.DataFrame) -> pd.DataFrame:
    if rank_ic_df.empty:
        return pd.DataFrame(columns=["Model", "Horizon", "RankIC_Mean", "RankIC_Std", "RankIC_IR", "N_Dates"])
    rows = []
    for (model, horizon), g in rank_ic_df.groupby(["Model", "Horizon"]):
        mean = g["RankIC"].mean()
        std = g["RankIC"].std(ddof=1)
        rows.append({
            "Model": model,
            "Horizon": int(horizon),
            "RankIC_Mean": float(mean),
            "RankIC_Std": float(std) if np.isfinite(std) else np.nan,
            "RankIC_IR": float(mean / std) if std and np.isfinite(std) else np.nan,
            "N_Dates": int(len(g)),
        })
    return pd.DataFrame(rows).sort_values(["Horizon", "RankIC_Mean"], ascending=[True, False]).reset_index(drop=True)


def load_predictions_if_exists(path: Path) -> pd.DataFrame | None:
    if path.exists():
        return pd.read_csv(path, parse_dates=["Date", "cutoff_date"])
    return None


def save_summary(predictions: pd.DataFrame, summary_path: Path) -> pd.DataFrame:
    summary = summarize_predictions_local(predictions)
    summary.to_csv(summary_path, index=False)
    return summary

## Classical Statistical Baselines

Replaces the Naive / Random Walk baseline with proper statistical models:

- **AutoARIMA** — automatic ARIMA order selection (AIC-based), the new primary baseline
- **AutoETS** — automatic Exponential Smoothing (Holt-Winters family)
- **AutoTheta** — Theta decomposition method (strong competition benchmark)

The original `Naive-RandomWalk` is retained as a sanity-check floor.

In [ ]:
import subprocess, sys, importlib
subprocess.check_call([sys.executable, "-m", "pip", "install", "statsforecast==1.7.4", "-q"])
importlib.invalidate_caches()  # force Python to see newly installed packages
import statsforecast
print(f"statsforecast {statsforecast.__version__}  ready")


statsforecast 1.7.4  ready


c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsforecast\core.py:26: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


: 

In [7]:
classical_path = RESULTS_DIR / "classical_predictions.csv"

# Classical statistical models — the new primary baselines replacing Naive.
# AutoARIMA picks the best ARIMA order automatically (AIC criterion).
# AutoETS selects the best Exponential Smoothing variant.
# AutoTheta applies the Theta decomposition method.
classical_predictions = run_classical_models(
    panel, config,
    models_to_run=("AutoARIMA", "AutoETS", "AutoTheta"),
)
classical_predictions.to_csv(classical_path, index=False)

# Keep Naive-RandomWalk as a floor reference (standard in academic comparisons).
naive_path = RESULTS_DIR / "naive_predictions.csv"
naive_predictions = run_naive_baseline(panel, config)
naive_predictions.to_csv(naive_path, index=False)

_combined = pd.concat([classical_predictions, naive_predictions], ignore_index=True)
classical_summary = _combined.groupby(["Model", "Horizon"]).apply(
    lambda g: pd.Series({
        "MAE":    g["y_true"].sub(g["y_pred"]).abs().mean(),
        "RMSE":   (g["y_true"].sub(g["y_pred"])**2).mean()**0.5,
        "DirAcc": ((g["y_true"] * g["y_pred"]) > 0).mean(),
        "N":      len(g),
    }),
    include_groups=False,
).reset_index()
classical_summary.to_csv(RESULTS_DIR / "classical_summary.csv", index=False)

print("Saved:", classical_path)
display(classical_summary)


Classical models horizon=1: 18 cutoffs  models=['AutoARIMA', 'AutoETS', 'AutoTheta']


: 

: 

## Neural models: N-BEATS and PatchTST

This section reuses `neural_predictions.csv` if it already exists and `FORCE_RERUN_NEURAL=False`.

In [ ]:
neural_path = RESULTS_DIR / "neural_predictions.csv"

if RUN_NEURAL_MODELS and neural_path.exists() and not FORCE_RERUN_NEURAL:
    print("Loading existing neural predictions:", neural_path)
    neural_predictions = load_predictions_if_exists(neural_path)
elif RUN_NEURAL_MODELS:
    from src.new_models import run_neuralforecast_models

    neural_predictions = run_neuralforecast_models(
        panel=panel,
        config=config,
        models_to_run=NEURAL_MODELS_TO_RUN,
    )
    neural_predictions.to_csv(neural_path, index=False)
else:
    neural_predictions = pd.DataFrame()
    print("Neural models disabled.")

if not neural_predictions.empty:
    neural_summary = save_summary(neural_predictions, RESULTS_DIR / "neural_summary.csv")
    display(neural_summary)
else:
    print("No neural predictions available.")

## Chronos zero-shot

This section reuses `chronos_predictions.csv` if it already exists and `FORCE_RERUN_CHRONOS=False`.

In [ ]:
chronos_path = RESULTS_DIR / "chronos_predictions.csv"

if RUN_CHRONOS and chronos_path.exists() and not FORCE_RERUN_CHRONOS:
    print("Loading existing Chronos predictions:", chronos_path)
    chronos_predictions = load_predictions_if_exists(chronos_path)
elif RUN_CHRONOS:
    from src.new_models import run_chronos_zero_shot

    chronos_predictions = run_chronos_zero_shot(
        panel=panel,
        config=config,
        model_id=CHRONOS_MODEL_ID,
    )
    chronos_predictions.to_csv(chronos_path, index=False)
else:
    chronos_predictions = pd.DataFrame()
    print("Chronos disabled.")

if not chronos_predictions.empty:
    chronos_summary = save_summary(chronos_predictions, RESULTS_DIR / "chronos_summary.csv")
    display(chronos_summary)
else:
    print("No Chronos predictions available.")

## Kronos zero-shot

Kronos is included and enabled by default in this notebook.

Before running this section, make sure:

- Your local Kronos repo exists at `KRONOS_LOCAL_REPO` from `src/config.py`.
- Kronos dependencies are installed.
- Your prepared data contains `open`, `high`, `low`, `close`, and `volume` columns. If `amount` is missing, the fixed `src/new_models.py` creates it from `close * volume`.

In [ ]:
kronos_path = RESULTS_DIR / "kronos_predictions.csv"

try:
    if RUN_KRONOS and kronos_path.exists() and not FORCE_RERUN_KRONOS:
        print("Loading existing Kronos predictions:", kronos_path)
        kronos_predictions = load_predictions_if_exists(kronos_path)
    elif RUN_KRONOS:
        print("Kronos local repo:", KRONOS_LOCAL_REPO)
        from src.new_models import run_kronos_zero_shot

        kronos_predictions = run_kronos_zero_shot(
            panel=panel,
            config=config,
            kronos_repo=KRONOS_LOCAL_REPO,
            model_id=KRONOS_MODEL_ID,
            tokenizer_id=KRONOS_TOKENIZER_ID,
        )
        kronos_predictions.to_csv(kronos_path, index=False)
    else:
        kronos_predictions = pd.DataFrame()
        print("Kronos disabled.")
except Exception as exc:
    kronos_predictions = pd.DataFrame()
    print("Kronos failed. This section is included, but your local Kronos setup needs fixing.")
    print("Error type:", type(exc).__name__)
    print("Error message:", exc)
    if not ALLOW_CONTINUE_IF_KRONOS_FAILS:
        raise

if not kronos_predictions.empty:
    kronos_summary = save_summary(kronos_predictions, RESULTS_DIR / "kronos_summary.csv")
    display(kronos_summary)
else:
    print("No Kronos predictions available yet.")

## Merge all predictions and create final metrics

In [ ]:
prediction_files = [
    RESULTS_DIR / "naive_predictions.csv",
    RESULTS_DIR / "classical_predictions.csv",
    RESULTS_DIR / "neural_predictions.csv",
    RESULTS_DIR / "chronos_predictions.csv",
    RESULTS_DIR / "kronos_predictions.csv",
]

frames = []
for path in prediction_files:
    if path.exists():
        df = pd.read_csv(path, parse_dates=["Date", "cutoff_date"])
        if not df.empty:
            frames.append(df)
            print("Loaded:", path.name, df.shape)
    else:
        print("Missing:", path.name)

if not frames:
    raise FileNotFoundError("No prediction files found. Run at least one model section first.")

all_predictions = pd.concat(frames, ignore_index=True)
all_predictions = all_predictions.dropna(subset=["Date", "Ticker", "Model", "Horizon", "y_true", "y_pred"])
all_predictions.to_csv(RESULTS_DIR / "all_predictions.csv", index=False)

model_summary = summarize_predictions_local(all_predictions)
model_summary.to_csv(RESULTS_DIR / "model_summary.csv", index=False)

rank_ic = cross_sectional_rank_ic_local(all_predictions, min_assets=MIN_ASSETS_FOR_RANKIC)
rank_ic.to_csv(RESULTS_DIR / "rank_ic_by_date.csv", index=False)

rank_ic_summary = summarize_rank_ic_local(rank_ic)
rank_ic_summary.to_csv(RESULTS_DIR / "rank_ic_summary.csv", index=False)

print("All predictions:", all_predictions.shape)
display(model_summary)
display(rank_ic_summary)

## Plots

In [ ]:
def save_grouped_bar(df: pd.DataFrame, value_col: str, title: str, filename: str, ylabel: str | None = None):
    if df.empty or value_col not in df.columns:
        print(f"Skipping {filename}: no data")
        return

    plot_df = df.copy()
    plot_df["Label"] = plot_df["Model"].astype(str) + " | h=" + plot_df["Horizon"].astype(str)
    plot_df = plot_df.sort_values(["Horizon", value_col])

    fig, ax = plt.subplots(figsize=(max(9, len(plot_df) * 0.7), 5))
    ax.bar(plot_df["Label"], plot_df[value_col])
    ax.set_title(title)
    ax.set_ylabel(ylabel or value_col)
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    out = PLOTS_DIR / filename
    plt.savefig(out, dpi=180)
    plt.show()
    print("Saved:", out)

save_grouped_bar(model_summary, "MAE", "MAE by Model and Horizon", "mae_by_model.png", "MAE")
save_grouped_bar(model_summary, "RMSE", "RMSE by Model and Horizon", "rmse_by_model.png", "RMSE")
save_grouped_bar(model_summary, "DirectionalAccuracy", "Directional Accuracy by Model and Horizon", "directional_accuracy_by_model.png", "Directional Accuracy")

if not rank_ic_summary.empty:
    save_grouped_bar(rank_ic_summary.rename(columns={"RankIC_Mean": "MeanRankIC"}), "MeanRankIC", "Mean RankIC by Model and Horizon", "rankic_by_model.png", "Mean RankIC")

In [ ]:
# Example actual vs predicted plot for the first available model/ticker/horizon.
example_df = all_predictions.sort_values(["Model", "Ticker", "Horizon", "Date"]).copy()

if not example_df.empty:
    ex_model = example_df["Model"].iloc[0]
    ex_ticker = example_df["Ticker"].iloc[0]
    ex_horizon = example_df["Horizon"].iloc[0]
    g = example_df[(example_df["Model"] == ex_model) & (example_df["Ticker"] == ex_ticker) & (example_df["Horizon"] == ex_horizon)].sort_values("Date")

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(g["Date"], g["y_true"], marker="o", label="Actual return")
    ax.plot(g["Date"], g["y_pred"], marker="o", label="Predicted return")
    ax.axhline(0, linewidth=1)
    ax.set_title(f"Actual vs Predicted Returns: {ex_model}, {ex_ticker}, h={ex_horizon}")
    ax.set_ylabel("Return")
    ax.legend()
    plt.tight_layout()
    out = PLOTS_DIR / "example_actual_vs_predicted.png"
    plt.savefig(out, dpi=180)
    plt.show()
    print("Saved:", out)
else:
    print("No predictions available for example plot.")

## Outputs created

This notebook creates or updates:

- `results/new_models/naive_predictions.csv`
- `results/new_models/naive_summary.csv`
- `results/new_models/neural_predictions.csv`
- `results/new_models/chronos_predictions.csv`
- `results/new_models/kronos_predictions.csv`
- `results/new_models/all_predictions.csv`
- `results/new_models/model_summary.csv`
- `results/new_models/rank_ic_by_date.csv`
- `results/new_models/rank_ic_summary.csv`
- `results/new_models/plots/*.png`

Use `model_summary.csv` and `rank_ic_summary.csv` in the final report/presentation.

In [ ]:
import pickle
with open(RESULTS_DIR / "emb_cache.pkl", "wb") as f:
    pickle.dump(embedding_predictions, f)
print("Cache saved!")

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from src.config import DATA_PROCESSED_DIR, RESULTS_DIR, PLOTS_DIR
from src.data_utils import load_prepared_panel
from src.new_models import BenchmarkConfig
from src.config import HORIZONS, INPUT_SIZE, TEST_STEP, MAX_TEST_DATES, MAX_STEPS, FREQ, RANDOM_SEED

panel = load_prepared_panel(DATA_PROCESSED_DIR)
panel = panel.sort_values(["Ticker", "Date"]).reset_index(drop=True)

config = BenchmarkConfig(
    horizons=HORIZONS,
    input_size=INPUT_SIZE,
    test_step=TEST_STEP,
    max_test_dates=MAX_TEST_DATES,
    max_steps=MAX_STEPS,
    freq=FREQ,
    random_seed=RANDOM_SEED,
)
print("Panel ready:", panel.shape)

In [ ]:
from src.embedding_models import run_nbeats_with_embeddings

embedding_predictions = run_nbeats_with_embeddings(
    panel=panel,
    config=config,
    chronos_model_id="amazon/chronos-t5-tiny",
    emb_dims=[8, 16, 32],
)
embedding_predictions.to_csv(RESULTS_DIR / "embedding_predictions.csv", index=False)
print(embedding_predictions["Model"].value_counts())

In [ ]:
print("NBEATS-NoEmb y_pred:")
print(embedding_predictions[embedding_predictions["Model"]=="NBEATS-NoEmb"]["y_pred"].describe())

print("\nNBEATS-Emb-8 y_pred:")
print(embedding_predictions[embedding_predictions["Model"]=="NBEATS-Emb-8"]["y_pred"].describe())

In [ ]:
embedding_predictions.to_csv(RESULTS_DIR / "embedding_predictions.csv", index=False)
print("Saved!")
print(embedding_predictions["Model"].value_counts())

if above three cells doesnt work, run last cells

In [ ]:
# # Reload embedding module fresh
# import importlib
# import src.embedding_models
# importlib.reload(src.embedding_models)
# from src.embedding_models import run_nbeats_with_embeddings

# embedding_predictions = run_nbeats_with_embeddings(
#     panel=panel,
#     config=config,
#     chronos_model_id="amazon/chronos-t5-tiny",
#     emb_dims=[8, 16, 32],
# )
# embedding_predictions.to_csv(RESULTS_DIR / "embedding_predictions.csv", index=False)
# print(embedding_predictions["Model"].value_counts())

In [ ]:
# print("NBEATS-NoEmb y_pred:")
# print(embedding_predictions[embedding_predictions["Model"]=="NBEATS-NoEmb"]["y_pred"].describe())

# print("\nNBEATS-Emb-8 y_pred:")
# print(embedding_predictions[embedding_predictions["Model"]=="NBEATS-Emb-8"]["y_pred"].describe())